# Pairwise Calibration (ELSST train split)
This notebook calibrates the cosine-similarity threshold of every model, and the Jaccard threshold of the
character 3-gram baseline, on the ELSST train split. The best thresholds are frozen in `scripts/config.py`.

In [ ]:
# project paths and the device come from scripts/config.py (works locally and on Colab)
import sys
import os
import gc
from pathlib import Path
import importlib

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import torch
from scripts import model_utils_pairwise as model_utils
from scripts import model_utils_shared, config
import warnings

# Reload config to get latest changes
importlib.reload(config)

# Silence warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Reproducibility + deterministic output paths
SEED = model_utils_shared.setup_reproducibility(config.SEED)

PROJECT_ROOT = Path(config.PROJECT_ROOT)
ELSST_CALLIBRATIONS_DIR = Path(config.CALIBRATION_DIR)
ELSST_PLOTS_DIR = Path(config.CALIBRATION_PLOTS_DIR)
ELSST_CALLIBRATIONS_DIR.mkdir(parents=True, exist_ok=True)
ELSST_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Calibration results: {ELSST_CALLIBRATIONS_DIR}")
print(f"Device: {config.DEVICE}")
print(f"Using seed: {SEED}")


## 1. Load Data
Load the ELSST train split (calibration data).

In [ ]:
# Calibration data: ELSST train split only (Parquet files with integer term ids).
# Negatives use all labels of both concepts. idx1 / idx2 point into `terms`.
data = model_utils_shared.load_pair_data(**config.DATASETS["elsst_train"])
terms = data["terms"]
idx1, idx2, labels = data["idx1"], data["idx2"], data["label"]
print(f"Total pairs: {len(labels):,} | positives: {int(labels.sum()):,} | unique terms: {len(terms):,}")


## 2. Models to Evaluate
We iterate through models defined in `config.py`.
Models are categorized as 'sentence' (SentenceTransformer) or 'token' (Transformer + Mean Pooling).

In [ ]:
models_to_test = config.MODELS
print("Models to be evaluated:")
for key, conf in models_to_test.items():
    print(f"- {conf['display_name']} ({key}) -> Type: {conf['type']}")

## 3. Analysis Loop
Iterate through each model:
1. Load the model and encode every term once.
2. Compute the cosine similarity of every pair.
3. Evaluate metrics across thresholds defined in config.
4. Generate visualizations.

In [ ]:
# the model only encodes the terms; similarities come from one term x term cosine matrix
all_summary_data = []
best_thresholds = {}

# Map raw names to display names for later
display_names_map = {k: v['display_name'] for k, v in config.MODELS.items()}
display_names_map[config.BASELINE_KEY] = config.BASELINE_DISPLAY_NAME

for model_name, model_conf in models_to_test.items():
    print(f"\n{'='*50}")
    print(f" Processing: {model_conf['display_name']}")
    print(f"{'='*50}")

    # Load with specific type (Sentence vs Token) on config.DEVICE
    model = model_utils_shared.load_model(model_name, model_type=model_conf['type'], device=config.DEVICE)
    if model is None:
        continue

    emb = model_utils_shared.build_embeddings(model, terms, batch_size=config.BATCH_SIZE)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    cos_terms = model_utils_shared.cosine_matrix(emb["norm_np"])
    similarities = model_utils_shared.pair_values(cos_terms, idx1, idx2)
    del cos_terms, emb
    gc.collect()

    current_model_results, best_threshold = model_utils.run_pairwise_optimization(
        model_name, similarities, labels, thresholds=config.PAIRWISE_PARAMS['thresholds']
    )
    all_summary_data.extend(current_model_results)
    best_thresholds[model_name] = best_threshold

    # Plot Individual
    model_utils.plot_individual_performance(
        current_model_results,
        model_name,
        best_threshold,
        str(ELSST_PLOTS_DIR),
        display_name=model_conf['display_name']
    )

## 3b. Character 3-gram Baseline
A method without a language model: two terms are matched when the Jaccard overlap of their letter trigrams is above a
threshold. Its threshold is calibrated in the same way, so it can be compared with the models.

In [ ]:
# Character 3-gram baseline, calibrated like the models
jaccard_terms = model_utils_shared.jaccard_matrix(terms, n=config.NGRAM_N)
similarities = model_utils_shared.pair_values(jaccard_terms, idx1, idx2)
del jaccard_terms

baseline_results, best_ngram_threshold = model_utils.run_pairwise_optimization(
    config.BASELINE_KEY, similarities, labels, thresholds=config.NGRAM_PARAMS['thresholds']
)
all_summary_data.extend(baseline_results)

model_utils.plot_individual_performance(
    baseline_results,
    config.BASELINE_KEY,
    best_ngram_threshold,
    str(ELSST_PLOTS_DIR),
    display_name=config.BASELINE_DISPLAY_NAME
)

## 4. Aggregate Results & Comparison
Compare all models in a single high-quality chart.

In [ ]:
summary_df = pd.DataFrame(all_summary_data)

csv_path = ELSST_CALLIBRATIONS_DIR / 'pairwise_results.csv'
summary_df.to_csv(csv_path, index=False)
print(f"\nDetailed results saved to: {csv_path}")

# Plot Comparison
model_utils.plot_comparison(summary_df, model_display_names=display_names_map, plots_dir=str(ELSST_PLOTS_DIR))

# Print Text Summary
model_utils.print_best_settings(summary_df)

# copy these lines into scripts/config.py
print("\nPaste into scripts/config.py:\n")
print("BEST_PAIRWISE_THRESHOLDS = {")
for model_name, t in best_thresholds.items():
    print(f'    "{model_name}": {t:.2f},')
print("}")
print(f"BEST_NGRAM_THRESHOLD = {best_ngram_threshold:.2f}")